__Random Forest model with nested cross-validation for tuning hyperparameters.__

## Settings

In [ ]:
path = '/home/jovyan/Main/mobility-bias_K'

In [ ]:
# period of interest
# April 2019 - pre COVID
period = '2019_04'

In [ ]:
# regions of interest
rois = ['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix', 
        'Philadelphia', 'San_Antonio', 'San_Diego', 'Dallas', 'San_Jose']

# leave one out
left_out = 'NYC'
# left_out = 'LA'
# left_out = 'Chicago'
# left_out = 'Houston'
# left_out = 'Phoenix'
# left_out = 'Philadelphia'
# left_out = 'San_Antonio'
# left_out = 'San_Diego'
# left_out = 'Dallas'
# left_out = 'San_Jose'

rois.remove(left_out)

In [ ]:
if int(period[:4]) < 2022:
    demo_year = period[:4]
else:
    demo_year = '2021'

In [ ]:
# k-fold cross validation params
CV_outer_folds = 10
CV_inner_folds = 3

## Setup

In [ ]:
import os
os.chdir(path)

In [ ]:
# !pip install statsmodels
# !pip install composition_stats
# !pip install seaborn
# !pip install shap

In [ ]:
import pandas as pd
import numpy as np
import math

import pickle
from bisect import insort

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score, make_scorer
import statsmodels.api as sm
import shap

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# plot layout settings
matplotlib.rcParams['font.size'] = 15
matplotlib.rcParams['xtick.labelsize'] = 13
matplotlib.rcParams['ytick.labelsize'] = 13
matplotlib.rcParams['axes.labelweight'] = 'bold'
matplotlib.rcParams['lines.linewidth'] = 2

# Load data 
Datapoints and demographics per census tract

In [ ]:
X = []
y = []
for roi in rois:
    file_name = f'{roi}_{period}'
    
    data = pd.read_csv(f'stats/demographics/{file_name}.csv')
    data = data.drop(['poptot', 'data_points_tot'], axis='columns')
    # data = data.sort_values('data_points', ignore_index=True)
    
    X.append(data.loc[:,'female':'other'])
    y.append(data.data_points)
    
X = pd.concat(X).reset_index(drop=True)
y = pd.concat(y).reset_index(drop=True)

## Fit model using nested cross validation 
Tune hyperparams and fit 10 models.  
Code largely inspired on: https://towardsdatascience.com/using-shap-with-cross-validation-d24af548fadc  

In [ ]:
# create parameter grid to for grid search (hyperparam tuning)
param_grid = {'max_depth': [10, 15, 20],
              'n_estimators': [200, 300, 400],
              'max_features': ['sqrt']}

In [ ]:
# enable use of sklearn.metrics.r2_score in GridSearchCV
custom_scorer = make_scorer(r2_score)

In [ ]:
# Establish CV scheme
CV_outer = KFold(n_splits=CV_outer_folds, shuffle=True, random_state=42)

ix_training, ix_test = [], []
# Loop through each fold and append the training & test indices to the empty lists above
for fold in CV_outer.split(X):
    ix_training.append(fold[0]), ix_test.append(fold[1])

In [ ]:
model_list = []
grid_searchst = []
R2_list = []
SHAP_values_per_fold = []
yhats = []
y_tests = []
y_preds = []
RMSE_list = []

## loop through each outer fold and extract SHAP values 
for i, (train_outer_ix, test_outer_ix) in enumerate(zip(ix_training, ix_test)):
    
    print('\n------ Fold Number:',i)
    
    # divide in train and test data by outer fold divide
    X_train, X_test = X.iloc[train_outer_ix, :], X.iloc[test_outer_ix, :]
    y_train, y_test = y.iloc[train_outer_ix], y.iloc[test_outer_ix]
    
    
    # HYPER-PARAM TUNING: (inner loop)
    # First create the base model to tune
    rf = RandomForestRegressor(random_state=None) 
    
    # CV_inner necessary, because GridSearchCV uses shuffle=False by default =.=
    CV_inner = KFold(n_splits=CV_inner_folds, shuffle=True, random_state=None) 
    
    # instantiate for hyperparam fit, using 10-fold cross-validation 
    search = GridSearchCV(rf, param_grid, cv=CV_inner, n_jobs=-1, scoring=custom_scorer)
    
    # fit all CV's of training data for finding best hyperparams
    # N.B: thus the training data is split int k folds internally; inner CV loop
    result = search.fit(X_train, y_train)
    print(result.best_estimator_, '\n')

    
    # MODEL FIT: (outer loop)
    # fit the whole training data of this fold to the model with these hyperparams
    result.best_estimator_.fit(X_train, y_train)
    
    # score model
    yhat = result.best_estimator_.predict(X_test)
    yhats.append(yhat)
    y_tests.append(y_test)
    
    model_list.append(result.best_estimator_)
    y_preds.append((yhat, y_test)) # for scatterplot 
    
    # use SHAP to explain predictions
    explainer = shap.TreeExplainer(result.best_estimator_)
    shap_values = explainer.shap_values(X_test)
    for SHAPs in shap_values: # change to SHAP values per CV if multiple CVs
        SHAP_values_per_fold.append(SHAPs) 

yhats_flat = np.concatenate(yhats, axis=0)
y_tests_flat = np.concatenate(y_tests, axis=0)

R2_skl = r2_score(y_tests_flat, yhats_flat)
R2_ols = sm.OLS(y_tests_flat, sm.add_constant(yhats_flat)).fit().rsquared

print('\n-----------------')
print(f'R2_skl = {round(R2_skl, 4)}')
print(f'R2_ols = {round(R2_ols, 4)}')

## Save model + outputs

In [ ]:
file_name = f'l1o{left_out}_{period}'

In [ ]:
# save model result (R2) as one line in R2 overview file
file = f'stats/model_outputs_l1o/R2s.txt'

newline = f'{file_name} R2_skl={round(R2_skl,4):.4f} R2_ols={round(R2_ols,4):.4f}'

try: 
    with open(file) as f:
        data = f.read().splitlines()
except:
    data = []

data = [newline if newline[:len(file_name)] in l else l for l in data]
if newline not in data: 
    insort(data, newline)

with open(file, 'w') as f:
    for l in data:
        f.write(l + '\n')

In [ ]:
# save model list
pickle.dump(model_list, open(f'stats/model_outputs_l1o/{file_name}', 'wb'))


# for prediction scatter plot
# save y predictions and actual values
y_preds = (yhats_flat, y_tests_flat)
pickle.dump(y_preds, open(f'stats/model_outputs_l1o/y_preds/{file_name}', 'wb'))


# for SHAP values plot
# save SHAP values
SHAP_values = pd.DataFrame(SHAP_values_per_fold, columns = X.columns)
SHAP_values.to_csv(f'stats/model_outputs_l1o/SHAP_values/{file_name}.csv', index=False)

# save X feature values with index matching SHAP values index
X_copy = X.copy()
X_copy = X_copy.reindex([ix for ix_test_fold in ix_test for ix in ix_test_fold]).reset_index(drop=True)
X_copy.to_csv(f'stats/model_outputs_l1o/X_feature_vals/{file_name}.csv', index=False)